In [10]:
import os
import json
import time
import requests
from pathlib import Path
from urllib.parse import unquote

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

SAVE_DIR = Path("menu_images")
SAVE_DIR.mkdir(exist_ok=True)

foods = {
   "butter chicken": "butter-chicken.jpg",
    "butter masala chicken": "butter-masala-chicken.jpg",

    "paneer butter masala": "paneer-butter-masala.jpg",
    "paneer makhani": "paneer-makhani.jpg",

    "palak paneer": "palak-paneer.jpg",

    "paneer tikka": "paneer-tikka.jpg",
    "paneer haryali tikka": "paneer-haryali-tikka.jpg",
    "paneer reshmi tikka": "paneer-reshmi-tikka.jpg",
    "paneer malai tikka": "paneer-malai-tikka.jpg",
    "paneer angara tikka": "paneer-angara-tikka.jpg",

    "chicken tikka": "chicken-tikka.jpg",
    "chicken reshmi tikka": "chicken-reshmi-tikka.jpg",
    "chicken boti tikka": "chicken-boti-tikka.jpg",
    "chicken bonanza tikka": "chicken-bonanza-tikka.jpg",
    "chicken angara tikka": "chicken-angara-tikka.jpg",
    "chicken afghani tikka": "chicken-afghani-tikka.jpg",
    "chicken hariyali tikka": "chicken-hariyali-tikka.jpg",

    "tandoori chicken": "tandoori-chicken.jpg",

    "chicken biryani": "chicken-biryani.jpg",
    "chicken hyderabadi dum biryani": "chicken-hyderabadi-biryani.jpg",

    "veg biryani": "veg-biryani.jpg",
    "dum veg biryani": "dum-veg-biryani.jpg",
    "paneer tikka biryani": "paneer-tikka-biryani.jpg",
    "veg hyderabadi biryani": "veg-hyderabadi-biryani.jpg",

    "mutton biryani": "mutton-biryani.jpg",

    "dal makhani": "dal-makhani.jpg",
    "dal fry": "dal-fry.jpg",
    "dal tadka": "dal-tadka.jpg",

    "malai kofta": "malai-kofta.jpg",
    "mughlai malai kofta": "mughlai-malai-kofta.jpg",

    "chicken seekh kebab": "chicken-seekh-kebab.jpg",
    "mutton seekh kebab": "mutton-seekh-kebab.jpg",

    "hara bhara kebab": "hara-bhara-kebab.jpg",

    "naan": "naan.jpg",
    "butter naan": "butter-naan.jpg",
    "garlic naan": "garlic-naan.jpg",
    "cheese garlic naan": "cheese-garlic-naan.jpg",

    "finger chips": "finger-chips.jpg",

    "onion pakoda": "onion-pakoda.jpg",
    "mix pakoda": "mix-pakoda.jpg",
    "aloo pakoda": "aloo-pakoda.jpg",
    "paneer pakoda": "paneer-pakoda.jpg",

    "paneer chilli": "paneer-chilli.jpg",
    "gobi chilli": "gobi-chilli.jpg",
    "chicken chilli": "chicken-chilli.jpg",

    "fish fry": "fish-fry.jpg",
    "fish masala": "fish-masala.jpg",

    "lassi": "lassi.jpg",

    "sweet corn soup": "sweet-corn-soup.jpg",
    "chicken sweet corn soup": "chicken-sweet-corn-soup.jpg",

    "tomato soup": "tomato-soup.jpg",

    "egg biryani": "egg-biryani.jpg",

    "chicken tikka masala": "chicken-tikka-masala.jpg",

    "shahi paneer": "shahi-paneer.jpg",

    "masala papad": "masala-papad.jpg",
    "roast papad": "roast-papad.jpg",

    "chana masala": "chana-masala.jpg",

    "veg pulao": "veg-pulao.jpg",
    "jeera rice": "jeera-rice.jpg",
    "garlic rice": "garlic-rice.jpg",

    "chicken curry": "chicken-curry.jpg",
    "chicken masala": "chicken-masala.jpg",

    "mutton curry": "mutton-curry.jpg",
    "mutton masala": "mutton-masala.jpg",
    "mutton bhuna": "mutton-bhuna.jpg",
}

options = Options()
# options.add_argument("--headless=new") 
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)

for query, filename in foods.items():
    try:
        filepath = SAVE_DIR / filename

        if filepath.exists():
            print(f"Skipping {filename} (Already exists)")
            continue

        print(f"Searching for: {query}")
        driver.get(f"https://www.bing.com/images/search?q={query.replace(' ', '+')}+High Quality image")

        # Wait for Bing's image result grid container to populate
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a.iusc")))
        
        # 'iusc' links contain metadata including the raw original high-res target image URL
        elements = driver.find_elements(By.CSS_SELECTOR, "a.iusc")
        
        high_res_url = None
        for elem in elements:
            try:
                # Bing stores metadata strings inside the 'm' attribute
                metadata_str = elem.get_attribute("m")
                if metadata_str:
                    metadata = json.loads(metadata_str)
                    # 'murl' points directly to the original high-resolution image asset host
                    if "murl" in metadata and metadata["murl"].startswith("http"):
                        high_res_url = metadata["murl"]
                        break
            except Exception:
                continue

        if high_res_url:
            print(f"Downloading high-res image: {high_res_url[:70]}...")
            
            response = requests.get(
                high_res_url, 
                timeout=15, 
                headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
            )
            
            # High-res files will comfortably clear a minimum size threshold
            if response.status_code == 200 and len(response.content) > 10000:
                with open(filepath, "wb") as f:
                    f.write(response.content)
                print(f"Saved sharp, high-res file -> {filename}")
            else:
                print(f"Failed to pull asset or size was too small (Status: {response.status_code})")
        else:
            print(f"Could not locate clear image metadata for {query}")

        time.sleep(2.5)

    except Exception as e:
        print(f"Error processing {query}: {str(e)[:120]}")

driver.quit()
print("High-res asset extraction complete.")

Searching for: butter chicken
Saved sharp, high-res file -> butter-chicken.jpg
Searching for: butter masala chicken
Saved sharp, high-res file -> butter-masala-chicken.jpg
Searching for: paneer butter masala
Saved sharp, high-res file -> paneer-butter-masala.jpg
Searching for: paneer makhani
Saved sharp, high-res file -> paneer-makhani.jpg
Searching for: palak paneer
Failed to pull asset or size was too small (Status: 403)
Searching for: paneer tikka
Saved sharp, high-res file -> paneer-tikka.jpg
Searching for: paneer haryali tikka
Saved sharp, high-res file -> paneer-haryali-tikka.jpg
Searching for: paneer reshmi tikka
Saved sharp, high-res file -> paneer-reshmi-tikka.jpg
Searching for: paneer malai tikka
Saved sharp, high-res file -> paneer-malai-tikka.jpg
Searching for: paneer angara tikka
Saved sharp, high-res file -> paneer-angara-tikka.jpg
Searching for: chicken tikka
Saved sharp, high-res file -> chicken-tikka.jpg
Searching for: chicken reshmi tikka
Saved sharp, high-res file ->